# ДЗ №3. Вторая ступень: проверка выреза предмета

Детектор плохо отличает предмет от похожего (AUC «предмет против трудного негатива»
Open Images около 0,75): поднять порог можно только ценой половины правильных находок.
Различение бутылки и банки — задача тонкой классификации, а детектор видит предмет
на всём кадре 640 px. Проверяем двухступенчатую схему без обучения:

1. детектор находит самую уверенную рамку класса;
2. MobileCLIP-S0 (как в ДЗ №2) оценивает **вырез** этой рамки: вероятность подписи
   класса среди подписей класса и четырёх похожих предметов (`photoquest_eval.verify`;
   списки похожих предметов зафиксированы до запуска).

Фото подтверждается, если score детектора выше порога COCO **и** вероятность подписи
выше порога второй ступени. Порог второй ступени подбирается на половине фото Open Images
под FAR на трудных негативах 40/20/10% и проверяется на другой половине, как в `03`.

**Критерий успеха (зафиксирован до запуска).** При FAR на трудных негативах 20%
(вход 640, среднее по классам) двухступенчатая схема сохраняет recall на Open Images
не ниже 70% и на COCO val2017 не ниже 65% (у одного детектора 56% и 47%).

**Настройки Kaggle:** GPU T4, Internet — On.
**Входные данные:** COCO 2017 (val) и вывод ноутбука обучения детектора на трудных негативах.

In [ ]:
import subprocess
import sys

# Ветка или коммит репозитория с пакетом photoquest_eval.
REPO_REF = "hw_3"
_ = subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"photoquest-eval[yolo] @ git+https://github.com/plekhovrodion/mlsd@{REPO_REF}",
    ]
)
REPO_COMMIT = subprocess.check_output(
    ["git", "ls-remote", "https://github.com/plekhovrodion/mlsd", REPO_REF], text=True
).split()[0]
print("photoquest_eval:", REPO_REF, REPO_COMMIT)

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import shutil  # noqa: F401 — ячейка общая для обоих ноутбуков
import time
import urllib.request
import zipfile
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from photoquest_eval.config import (  # noqa: F401 — ячейка общая для обоих ноутбуков
    HOME_CLASSES,
    IMAGE_SIZES,
    SEED,
    TARGETS,
    YOLO_SETTINGS,
)

STARTED_UTC = datetime.now(UTC).isoformat()
STARTED = time.perf_counter()
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle Settings → Accelerator → GPU T4")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def package_versions(*names):
    return {name: importlib.metadata.version(name) for name in names}

In [ ]:
_ = subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "open_clip_torch==4.2.1",
        "timm==1.0.31",
        "ftfy==6.3.1",
    ]
)
MOBILECLIP_COMMIT = "48faa0fea4b08d74188b3841771aca6ff2c92852"
_ = subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        f"git+https://github.com/apple/ml-mobileclip.git@{MOBILECLIP_COMMIT}",
    ]
)

In [ ]:
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import roc_auc_score

from photoquest_eval.calibration import threshold_for_far
from photoquest_eval.data import (
    OPENIMAGES_URLS,
    copy_images,
    download_openimages,
    load_coco,
    load_openimages,
    sample_openimages,
)
from photoquest_eval.inference import predict_best_boxes
from photoquest_eval.tradeoff import mean_rates, status_rates, two_fold_tradeoff
from photoquest_eval.verify import LOOKALIKES, crop_box, prompts, target_probability

OUTPUT = Path("/kaggle/working/hw3_verify")
OUTPUT.mkdir(parents=True, exist_ok=True)
OI_SAMPLE = {"positives_per_class": 400, "negatives_per_class": 800, "seed": SEED}
IMAGE_SIZE = 640
HN_LEVELS = (0.40, 0.20, 0.10)
CROP_MARGIN = 0.1
MOBILECLIP = (
    "https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt",
    "809b408eff74f8058843e86a1f92967097d42ba782450e85b8f4867b7f0ca0b7",
)

## Данные — те же выборки, что в `03_evaluate_models.ipynb`

In [ ]:
import kagglehub

COCO_ROOT = Path(kagglehub.dataset_download("nikitamanaenkov/microsoft-coco-2017"))
VAL_ANNOTATIONS = sorted(COCO_ROOT.rglob("instances_val2017.json"))[0]
assert sha256(VAL_ANNOTATIONS) == (
    "e8c7f7908f1d7278341fae127d0da654f102f11bd7b21d8aeefa635b8c810b6f"
), "Снимок аннотаций отличается от ДЗ №2"
val_root = VAL_ANNOTATIONS.parent.parent
INPUT_VAL_IMAGES = next(
    p for p in (val_root / "val2017", val_root / "images" / "val2017") if p.is_dir()
)
# Оценка проходит по val много раз (модели × 3 размера), поэтому оригиналы один раз
# копируются на локальный диск без изменений, как в ДЗ №2.
VAL_IMAGES = Path("/tmp/coco_val2017")
failed_copies = copy_images(sorted(INPUT_VAL_IMAGES.glob("*.jpg")), VAL_IMAGES)
assert not failed_copies, failed_copies[:5]
coco = load_coco(VAL_ANNOTATIONS, VAL_IMAGES, TARGETS)
print(
    len(coco.image_ids),
    "фото;",
    "положительных по классам:",
    dict(zip(TARGETS, coco.labels.sum(axis=0).tolist(), strict=True)),
)

In [ ]:
OI_DIR = Path("/kaggle/working/openimages")
OI_DIR.mkdir(exist_ok=True)
csv_paths = {}
for key, name in (
    ("image_labels", "test-human-imagelabels.csv"),
    ("boxes", "test-annotations-bbox.csv"),
):
    csv_paths[key] = OI_DIR / name
    if not csv_paths[key].exists():
        urllib.request.urlretrieve(OPENIMAGES_URLS[key], csv_paths[key])
META_URL = "https://storage.googleapis.com/openimages/2018_04/test/test-images-with-rotation.csv"
csv_paths["meta"] = OI_DIR / "test-images-with-rotation.csv"
if not csv_paths["meta"].exists():
    urllib.request.urlretrieve(META_URL, csv_paths["meta"])
{key: sha256(path) for key, path in csv_paths.items()}

In [ ]:
oi_ids = sample_openimages(csv_paths["image_labels"], TARGETS, **OI_SAMPLE)
failed = download_openimages(oi_ids, OI_DIR / "images")
print(f"Выбрано {len(oi_ids):,} фото, не скачалось {len(failed)}")
oi = load_openimages(csv_paths["image_labels"], csv_paths["boxes"], OI_DIR / "images", TARGETS)
oi_meta = pd.read_csv(csv_paths["meta"]).set_index("ImageID")
manifest = oi_meta.reindex(oi.image_ids)[
    ["License", "Author", "OriginalLandingURL", "Rotation"]
].reset_index(names="ImageID")
manifest.to_csv(OUTPUT / "openimages_manifest.csv", index=False)
known = oi.labels >= 0
display(
    pd.DataFrame(
        {
            "positive_images": (oi.labels == 1).sum(axis=0),
            "verified_negatives": (oi.labels == 0).sum(axis=0),
            "unknown": (~known).sum(axis=0),
        },
        index=TARGETS,
    )
)
print("Фото с ненулевым Rotation:", int(manifest["Rotation"].fillna(0).ne(0).sum()))

## Первая ступень: детектор

In [ ]:
DETECTOR = None
for name in ("yolo11s_hardneg", "yolo11n_hardneg_v2", "yolo11n_hardneg"):
    found = sorted(Path("/kaggle/input").rglob(f"{name}.pt"))
    if found:
        DETECTOR, DETECTOR_PATH = name, found[0]
        break
if DETECTOR is None:
    raise FileNotFoundError("Добавьте во входные данные вывод ноутбука 04 или 05")
print("Детектор:", DETECTOR, sha256(DETECTOR_PATH))
det = {}
for dataset in (coco, oi):
    started = time.perf_counter()
    det[dataset.name] = predict_best_boxes(
        DETECTOR_PATH, dataset.image_paths, TARGETS, IMAGE_SIZE, device=0, **YOLO_SETTINGS
    )
    print(dataset.name, f"{time.perf_counter() - started:.0f} с")
floors = []
for column in range(len(TARGETS)):
    negatives = det["coco"][0][coco.labels[:, column] == 0, column]
    floors.append(threshold_for_far(negatives, "confidence_95pct", 0.05, 0.95))
pd.Series(floors, index=TARGETS, name="coco_cutoff")

## Вторая ступень: MobileCLIP по вырезам

In [ ]:
import mobileclip
from torch.utils.data import DataLoader, Dataset

clip_path = OUTPUT / "pretrained" / "mobileclip_s0.pt"
clip_path.parent.mkdir(parents=True, exist_ok=True)
if not clip_path.exists():
    urllib.request.urlretrieve(MOBILECLIP[0], clip_path)
assert sha256(clip_path) == MOBILECLIP[1], "Не совпал SHA-256 MobileCLIP-S0"
clip, _, preprocess = mobileclip.create_model_and_transforms(
    "mobileclip_s0", pretrained=str(clip_path), reparameterize=True, device="cuda"
)
clip.eval()
tokenizer = mobileclip.get_tokenizer("mobileclip_s0")
with torch.inference_mode():
    text_features = {
        target: F.normalize(clip.encode_text(tokenizer(prompts(target)).to("cuda")).float(), dim=-1)
        for target in TARGETS
    }


class Crops(Dataset):
    """Вырезы лучших рамок: (фото, класс) для всех пар, где детектор что-то нашёл."""

    def __init__(self, paths, boxes):
        self.paths, self.boxes = paths, boxes
        self.pairs = [
            (row, column)
            for row in range(len(paths))
            for column in range(len(TARGETS))
            if np.isfinite(boxes[row, column]).all()
        ]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        row, column = self.pairs[index]
        with Image.open(self.paths[row]) as image:
            image = image.convert("RGB")
            area = crop_box(image.width, image.height, self.boxes[row, column], CROP_MARGIN)
            crop = image.crop(area) if area else image
        return preprocess(crop), row, column


def verify(dataset):
    boxes = det[dataset.name][1]
    probability = np.zeros(boxes.shape[:2], dtype=np.float32)
    loader = DataLoader(Crops(dataset.image_paths, boxes), batch_size=256, num_workers=4)
    for batch, rows, columns in loader:
        with torch.inference_mode():
            features = F.normalize(clip.encode_image(batch.to("cuda")).float(), dim=-1)
        for feature, row, column in zip(features, rows.tolist(), columns.tolist(), strict=True):
            target = TARGETS[column]
            similarity = (feature @ text_features[target].T).cpu().numpy()
            probability[row, column] = target_probability(similarity)
    return probability


clip_scores = {}
clip_timing = {}
for dataset in (coco, oi):
    started = time.perf_counter()
    clip_scores[dataset.name] = verify(dataset)
    clip_timing[dataset.name] = time.perf_counter() - started
    print(dataset.name, f"{clip_timing[dataset.name]:.0f} с")
np.savez_compressed(
    OUTPUT / "two_stage_scores.npz",
    **{f"{name}__detector": det[name][0] for name in det},
    **{f"{name}__boxes": det[name][1] for name in det},
    **{f"{name}__clip": clip_scores[name] for name in clip_scores},
)

## Результат

Для каждого класса сравниваются три правила при одном и том же целевом FAR
на трудных негативах: только детектор; только вторая ступень (среди фото,
где детектор выше порога COCO); их произведение как единый score.

In [ ]:
fold = np.random.default_rng(SEED).random(len(oi.image_ids)) < 0.5
rows, status_records = [], []
for column, target in enumerate(TARGETS):
    labels = oi.labels[:, column]
    known = labels >= 0
    floor = floors[column]
    rules = {
        "detector": (det["openimages"][0][:, column], det["coco"][0][:, column], floor),
        # Вторая ступень работает только на фото, прошедших порог детектора.
        "two_stage": (
            np.where(
                det["openimages"][0][:, column] > floor, clip_scores["openimages"][:, column], -1.0
            ),
            np.where(det["coco"][0][:, column] > floor, clip_scores["coco"][:, column], -1.0),
            -0.5,
        ),
        "product": (
            det["openimages"][0][:, column] * clip_scores["openimages"][:, column],
            det["coco"][0][:, column] * clip_scores["coco"][:, column],
            0.0,
        ),
    }
    for rule, (oi_s, coco_s, rule_floor) in rules.items():
        row = {
            "target": target,
            "rule": rule,
            "oi_auc_hardneg": roc_auc_score(labels[known], oi_s[known]),
        }
        for point in two_fold_tradeoff(labels, oi_s, rule_floor, HN_LEVELS, fold):
            tag = int(point.level * 100)
            coco_recall, coco_far = mean_rates(coco.labels[:, column], coco_s, point.cutoffs)
            row |= {
                f"oi_recall@hn{tag}": point.recall,
                f"oi_far@hn{tag}": point.far,
                f"coco_recall@hn{tag}": coco_recall,
                f"coco_far@hn{tag}": coco_far,
            }
        rows.append(row)
    # Три статуса двухступенчатой схемы: «да» — выше порога HN 20%, «не уверен» — прошёл
    # только детектор, «не найдено» — детектор ниже порога COCO.
    oi_s, coco_s, _ = rules["two_stage"]
    point = two_fold_tradeoff(labels, oi_s, -0.5, (0.20,), fold)[0]
    high = float(np.mean(point.cutoffs))
    for name, values in (
        ("oi_positive", oi_s[labels == 1]),
        ("oi_hardneg", oi_s[labels == 0]),
        ("coco_positive", coco_s[coco.labels[:, column] == 1]),
        ("coco_negative", coco_s[coco.labels[:, column] == 0]),
    ):
        confirmed, uncertain, not_found = status_rates(values, -0.5, high)
        status_records.append(
            {
                "target": target,
                "set": name,
                "confirmed": confirmed,
                "uncertain": uncertain,
                "not_found": not_found,
            }
        )
two_stage = pd.DataFrame(rows)
two_stage.to_csv(OUTPUT / "two_stage_tradeoff_per_class.csv", index=False)
statuses = pd.DataFrame(status_records)
statuses.to_csv(OUTPUT / "two_stage_statuses_per_class.csv", index=False)
display(two_stage.groupby("rule", sort=False).mean(numeric_only=True).round(3).T)
display(two_stage.pivot(index="target", columns="rule", values="oi_auc_hardneg").round(3))
display(two_stage.pivot(index="target", columns="rule", values="oi_recall@hn20").round(3))
display(statuses.groupby("set").mean(numeric_only=True).round(3))

In [ ]:
summary = two_stage.groupby("rule").mean(numeric_only=True)
criterion = {
    "oi_recall@hn20": float(summary.loc["two_stage", "oi_recall@hn20"]),
    "coco_recall@hn20": float(summary.loc["two_stage", "coco_recall@hn20"]),
}
criterion["passed"] = criterion["oi_recall@hn20"] >= 0.70 and criterion["coco_recall@hn20"] >= 0.65
crops = {name: int(np.isfinite(det[name][1]).all(axis=-1).sum()) for name in det}
metadata = {
    "repo_ref": REPO_REF,
    "repo_commit": REPO_COMMIT,
    "detector": DETECTOR,
    "detector_sha256": sha256(DETECTOR_PATH),
    "mobileclip_sha256": MOBILECLIP[1],
    "mobileclip_commit": MOBILECLIP_COMMIT,
    "lookalikes": {target: list(LOOKALIKES[target]) for target in TARGETS},
    "crop_margin": CROP_MARGIN,
    "image_size": IMAGE_SIZE,
    "hn_levels": list(HN_LEVELS),
    "criterion": criterion,
    "crops": crops,
    "clip_seconds": clip_timing,
    "clip_ms_per_crop_t4": {name: 1000 * clip_timing[name] / max(crops[name], 1) for name in crops},
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "packages": package_versions("torch", "ultralytics", "open_clip_torch", "timm", "numpy"),
    "started_utc": STARTED_UTC,
    "elapsed_seconds": round(time.perf_counter() - STARTED, 1),
}
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
with zipfile.ZipFile("/kaggle/working/hw3_verify_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file() and "pretrained" not in path.relative_to(OUTPUT).parts:
            z.write(path, path.relative_to(OUTPUT))
print(json.dumps(metadata, indent=2, ensure_ascii=False))